# 07 · Why no cleaning effect? Exploratory checks
**Exploratory, not pre-registered** (main test: https://osf.io/6bsek). These checks try to explain the main result (+7.3%, p = 0.97).

1. Was pollution already rising before each action (pre-trend)?
2. Were actions taken at unusual times (before-window unusually clean or dirty)? If so, part of the +7.3% may be a return to normal.
3. Does season explain it (month-matched placebo)?
4. Do courts act when *raw* (headline) PM2.5 is high?

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
RAW = "https://raw.githubusercontent.com/prantiqueroy/dhaka-air-enforcement/main/data/"
d  = pd.read_csv(RAW + "dhaka_daily_adjusted.csv", index_col=0, parse_dates=True)
tl = pd.read_csv(RAW + "enforcement_timeline.csv")
ne = pd.read_csv(RAW + "natural_experiments.csv")
tl["date"] = pd.to_datetime(tl["date"], dayfirst=True, format="mixed")
ne["start_date"] = pd.to_datetime(ne["start_date"], dayfirst=True, format="mixed")
for event, day in {"Eid ul-Fitr 2016": "2016-07-07", "Eid ul-Fitr 2018": "2018-06-16", "Eid ul-Fitr 2024": "2024-04-11"}.items():
    ne.loc[ne["event"] == event, "start_date"] = pd.Timestamp(day)

# same setup as notebook 06
eid_days = ne.loc[ne["event"].str.startswith("Eid"), "start_date"]
s = d["adjusted_pct"].copy()
for e in eid_days:
    s.loc[e - pd.Timedelta(days=1): e + pd.Timedelta(days=3)] = np.nan
s.loc["2020-03-26":"2020-05-30"] = np.nan
s = s.dropna()

FROZEN = ["E01","E02","E03","E04","E08","E10","E11","E12","E14","E15","E24","E25"]
ev = tl[tl["event_id"].isin(FROZEN)].sort_values("date")[["event_id", "date", "tool"]]
ev["cluster"] = (ev["date"].diff().dt.days.fillna(999) >= 30).cumsum()
clusters = ev.groupby("cluster").agg(date=("date", "first"), events=("event_id", "+".join))

def mean_in(series, day, a, b, need=10):
    x = series.loc[day + pd.Timedelta(days=a): day + pd.Timedelta(days=b)]
    return x.mean() if len(x) >= need else np.nan

avoid = pd.concat([tl["date"], eid_days, pd.Series(pd.date_range("2020-03-26", "2020-05-30"))])
cand = d.index[(d.index > "2016-05-01") & (d.index < "2025-02-01")]
dist = np.array([np.abs((avoid.values - np.datetime64(c)).astype("timedelta64[D]").astype(int)).min() for c in cand])
pool = cand[dist >= 40]
print("Clusters:", len(clusters), "| Random comparison dates:", len(pool))

In [ ]:
clusters["level_before"] = [mean_in(s, x, -30, -1) for x in clusters["date"]]
clusters["level_after"]  = [mean_in(s, x, 1, 30) for x in clusters["date"]]
clusters["trend_before"] = [mean_in(s, x, -30, -1) - mean_in(s, x, -60, -31) for x in clusters["date"]]
print(clusters.round(1).to_string())

r_level = np.nanmean([mean_in(s, pd.Timestamp(x), -30, -1) for x in pool])
r_trend = [mean_in(s, pd.Timestamp(x), -30, -1) - mean_in(s, pd.Timestamp(x), -60, -31) for x in pool]
print("\nWeather-adjusted level, 30 days BEFORE actions:", round(clusters["level_before"].mean(), 1), "%")
print("Weather-adjusted level, 30 days AFTER actions: ", round(clusters["level_after"].mean(), 1), "%")
print("Typical level (30 days before random dates):   ", round(r_level, 1), "%")
print("Pre-trend before actions:", round(clusters["trend_before"].mean(), 1), "% | random dates:", round(np.nanmean(r_trend), 1), "% (SD", round(np.nanstd(r_trend), 1), ")")

In [ ]:
plt.figure(figsize=(8, 3.5))
x = np.arange(len(clusters))
plt.bar(x - 0.2, clusters["level_before"], 0.4, label="30 days before", color="lightsteelblue")
plt.bar(x + 0.2, clusters["level_after"], 0.4, label="30 days after", color="steelblue")
plt.axhline(r_level, color="black", ls="--", lw=1, label="Typical level")
plt.xticks(x, clusters["events"], rotation=60); plt.ylabel("% vs weather-expected")
plt.title("Weather-adjusted PM2.5 before and after each action"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
def effect(day):
    b = s.loc[day - pd.Timedelta(days=30): day - pd.Timedelta(days=1)]
    a = s.loc[day + pd.Timedelta(days=1): day + pd.Timedelta(days=30)]
    return a.mean() - b.mean() if len(a) >= 15 and len(b) >= 15 else np.nan

clusters["effect"] = [effect(x) for x in clusters["date"]]
allp = pd.Series({pd.Timestamp(x): effect(pd.Timestamp(x)) for x in pool}).dropna()
bymonth = {m: g.values for m, g in allp.groupby(allp.index.month)}
rng = np.random.default_rng(1)
sims = np.array([np.mean([rng.choice(bymonth[t.month]) for t in clusters["date"]]) for _ in range(2000)])
print("Observed average effect:", round(clusters["effect"].mean(), 1), "%")
print("Random dates in the SAME months: average", round(sims.mean(), 1), "%")
print("Share of month-matched random sets as high or higher:", round((sims >= clusters["effect"].mean()).mean(), 3))

In [ ]:
mon_raw = d["pm25"].groupby(d.index.month).mean()
def raw_vs_month(day):
    return mean_in(d["pm25"], day, -30, -1) / mon_raw[(day - pd.Timedelta(days=15)).month] * 100 - 100
ev_raw = np.nanmean([raw_vs_month(x) for x in clusters["date"]])
rd_raw = np.nanmean([raw_vs_month(pd.Timestamp(x)) for x in pool])
print("Raw PM2.5 in the 30 days before actions, vs normal for that month:", round(ev_raw, 1), "%")
print("Same for random dates:", round(rd_raw, 1), "%")

In [ ]:
clusters.to_csv("why_checks.csv")
print("Saved")